In [1]:
# ============================================================
# AMAZON ML CHALLENGE 2026
# M2 — BLOCKING & CANDIDATE RETRIEVAL
# STAGE 1 — CONTROLLED 100K EXPERIMENT
# ============================================================

from pathlib import Path
from collections import defaultdict
from time import perf_counter
import re
import unicodedata
import random
import gc
import json

import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

RANDOM_SEED = 42

# Main experiment size
S1_SAMPLE_SIZE = 100_000

# Process Source-1 records in batches to reduce RAM pressure
BATCH_SIZE = 5_000

# Maximum number of candidate IDs retained per S1 entity.
# IMPORTANT:
# None = no artificial candidate limit.
MAX_CANDIDATES = None

# Save detailed candidate provenance
SAVE_PROVENANCE = True

# ============================================================
# Project directories
# ============================================================

def find_project_root():
    """
    Locate project root by searching upward for dataset/train.
    """
    current = Path.cwd().resolve()

    candidates = [current] + list(current.parents)

    for path in candidates:
        if (path / "dataset" / "train").exists():
            return path

    # Fallback
    return current


PROJECT_ROOT = find_project_root()

TRAIN_DIR = PROJECT_ROOT / "dataset" / "train"

M2_DIR = PROJECT_ROOT / "m2_work"

OUTPUT_DIR = M2_DIR / "outputs" / "blocking_100k"
REPORT_DIR = M2_DIR / "reports"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


print("=" * 80)
print("M2 — BLOCKING & CANDIDATE RETRIEVAL")
print("STAGE 1 — CONTROLLED 100K EXPERIMENT")
print("=" * 80)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nTraining directory:")
print(TRAIN_DIR)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nBatch size:")
print(f"{BATCH_SIZE:,}")

print("\nS1 sample size:")
print(f"{S1_SAMPLE_SIZE:,}")


# ============================================================
# 2. FILE PATHS
# ============================================================

S1_PATH = TRAIN_DIR / "train_source1.tsv"
S2_PATH = TRAIN_DIR / "train_source2.tsv"
S3_PATH = TRAIN_DIR / "train_source3.tsv"
GT_PATH = TRAIN_DIR / "train_ground_truth.tsv"


required_files = [
    S1_PATH,
    S2_PATH,
    S3_PATH,
    GT_PATH,
]

print("\nChecking files...")

for path in required_files:
    print(f"{path.name:<30} ", end="")

    if not path.exists():
        print("MISSING")
        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

    print("OK")


# ============================================================
# 3. DATA LOADING
# ============================================================

def load_tsv(path):
    return pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False
    )


print("\nLoading Source 1...")

s1 = load_tsv(S1_PATH)

print(f"Source 1: {s1.shape}")

print("\nLoading Source 2...")

s2 = load_tsv(S2_PATH)

print(f"Source 2: {s2.shape}")

print("\nLoading Source 3...")

s3 = load_tsv(S3_PATH)

print(f"Source 3: {s3.shape}")

print("\nLoading ground truth...")

gt = load_tsv(GT_PATH)

print(f"Ground truth: {gt.shape}")


# ============================================================
# 4. VALIDATE SCHEMA
# ============================================================

SOURCE_COLUMNS = {
    "entity_id",
    "business_name",
    "business_address",
    "country",
}

GT_COLUMNS = {
    "source1_entity_id",
    "matched_entity_ids",
}


for name, df in {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3,
}.items():

    missing = SOURCE_COLUMNS - set(df.columns)

    if missing:
        raise ValueError(
            f"{name} missing columns: {missing}"
        )


missing_gt = GT_COLUMNS - set(gt.columns)

if missing_gt:
    raise ValueError(
        f"Ground truth missing columns: {missing_gt}"
    )


# ============================================================
# 5. BASIC ID VALIDATION
# ============================================================

print("\nID validation...")

for name, df in {
    "S1": s1,
    "S2": s2,
    "S3": s3,
}.items():

    duplicate_count = df["entity_id"].duplicated().sum()

    print(
        f"{name}: "
        f"{len(df):,} rows | "
        f"{df['entity_id'].nunique():,} unique IDs | "
        f"{duplicate_count:,} duplicate IDs"
    )

    if duplicate_count:
        raise ValueError(
            f"{name} contains duplicate entity IDs."
        )


# ============================================================
# 6. TEXT NORMALIZATION HELPERS
# ============================================================

def safe_text(value):
    if value is None:
        return ""

    return str(value)


def unicode_normalize(value):
    value = safe_text(value)

    return unicodedata.normalize(
        "NFKC",
        value
    )


def casefold_text(value):
    return safe_text(value).casefold()


def collapse_whitespace(value):
    return re.sub(
        r"\s+",
        " ",
        safe_text(value).strip()
    )


def punctuation_normalize(value):
    value = unicode_normalize(value)

    chars = []

    for char in value:

        category = unicodedata.category(char)

        if category.startswith("P"):
            chars.append(" ")
        else:
            chars.append(char)

    return "".join(chars)


def alphanumeric_normalize(value):
    value = casefold_text(value)

    return re.sub(
        r"[^a-zA-Z0-9]+",
        "",
        value
    )


def tokenize(value):
    value = casefold_text(value)

    return re.findall(
        r"\w+",
        value,
        flags=re.UNICODE
    )


def sorted_tokens(value):
    tokens = tokenize(value)

    if not tokens:
        return ""

    return " ".join(sorted(tokens))


def digit_signature(value):
    """
    Auxiliary numeric signature only.

    IMPORTANT:
    Empty digit signatures are NOT used as blocking keys.
    """

    value = safe_text(value)

    digits = re.findall(
        r"\d+",
        value
    )

    if not digits:
        return ""

    return "".join(digits)


# ============================================================
# 7. NORMALIZED REPRESENTATION GENERATOR
# ============================================================

def normalize_series(series):
    """
    Produce the same conceptual representation family
    validated during M1.

    We intentionally do not use business-name digits
    as a blocking key.
    """

    result = pd.DataFrame(index=series.index)

    result["original"] = series.map(safe_text)

    result["unicode"] = result["original"].map(
        unicode_normalize
    )

    result["casefold"] = result["unicode"].map(
        casefold_text
    )

    result["whitespace"] = result["casefold"].map(
        collapse_whitespace
    )

    result["punctuation"] = result["whitespace"].map(
        punctuation_normalize
    )

    result["alphanumeric"] = result["punctuation"].map(
        alphanumeric_normalize
    )

    result["sorted_tokens"] = result["punctuation"].map(
        sorted_tokens
    )

    result["digits"] = result["original"].map(
        digit_signature
    )

    return result


# ============================================================
# 8. BLOCKING KEY GENERATORS
# ============================================================

def clean_key(value):
    value = safe_text(value).strip()

    if not value:
        return None

    return value


def first_numeric_token(address):
    """
    Extract first numeric component from address.
    """

    address = safe_text(address)

    match = re.search(
        r"\b\d+[A-Za-z]?\b",
        address
    )

    if not match:
        return None

    return match.group(0).casefold()


def address_token_signature(address):
    """
    Conservative address token signature.

    Uses:
      house number
      selected address tokens

    Does NOT create a key from an empty representation.
    """

    tokens = tokenize(address)

    if not tokens:
        return None

    house_number = first_numeric_token(address)

    if house_number is None:
        return None

    # Remove very short/noisy tokens.
    useful_tokens = [
        token
        for token in tokens
        if len(token) >= 3
        and token != house_number
    ]

    if not useful_tokens:
        return house_number

    # Keep only a small deterministic signature.
    useful_tokens = sorted(
        set(useful_tokens)
    )[:4]

    return (
        house_number
        + "|"
        + "|".join(useful_tokens)
    )


def city_like_token_signature(address):
    """
    Lightweight address token signature.

    This is intentionally broad and will be evaluated
    experimentally rather than assumed to be safe.
    """

    tokens = tokenize(address)

    if len(tokens) < 2:
        return None

    useful = [
        token
        for token in tokens
        if len(token) >= 4
    ]

    if not useful:
        return None

    # Use last useful token(s), because address strings
    # frequently end with locality/state/country information.
    return "|".join(
        sorted(set(useful[-2:]))
    )


# ============================================================
# 9. BLOCK DEFINITIONS
# ============================================================

BLOCK_NAMES = [
    "name_exact",
    "name_casefold",
    "name_sorted_tokens",
    "address_exact",
    "address_sorted_tokens",
    "address_house_signature",
    "name_address_combined",
    "country_name_casefold",
]


# ============================================================
# 10. PREPARE SOURCE REPRESENTATIONS
# ============================================================

print("\nPreparing normalized representations...")

prep_start = perf_counter()


def prepare_source(df, source_name):

    work = df[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country",
        ]
    ].copy()

    work["entity_id"] = work["entity_id"].astype(str)

    print(
        f"\nNormalizing {source_name}..."
    )

    name_norm = normalize_series(
        work["business_name"]
    )

    address_norm = normalize_series(
        work["business_address"]
    )

    work["name_original"] = name_norm["original"]
    work["name_casefold"] = name_norm["casefold"]
    work["name_sorted_tokens"] = name_norm["sorted_tokens"]

    work["address_original"] = address_norm["original"]
    work["address_casefold"] = address_norm["casefold"]
    work["address_sorted_tokens"] = address_norm["sorted_tokens"]

    work["name_digits"] = name_norm["digits"]
    work["address_digits"] = address_norm["digits"]

    work["address_house_signature"] = (
        work["business_address"]
        .map(address_token_signature)
    )

    work["address_city_signature"] = (
        work["business_address"]
        .map(city_like_token_signature)
    )

    work["country_key"] = (
        work["country"]
        .map(safe_text)
        .map(casefold_text)
        .map(clean_key)
    )

    work["name_address_combined"] = (
        work["name_casefold"]
        + "||"
        + work["address_casefold"]
    )

    work["country_name_casefold"] = (
        work["country_key"].fillna("")
        + "||"
        + work["name_casefold"].fillna("")
    )

    return work


s1_p = prepare_source(
    s1,
    "Source 1"
)

s2_p = prepare_source(
    s2,
    "Source 2"
)

s3_p = prepare_source(
    s3,
    "Source 3"
)


print(
    f"\nRepresentation preparation completed in "
    f"{perf_counter() - prep_start:.2f} seconds."
)


# ============================================================
# 11. SAMPLE SOURCE 1
# ============================================================

rng = np.random.default_rng(
    RANDOM_SEED
)

sample_size = min(
    S1_SAMPLE_SIZE,
    len(s1_p)
)

sample_indices = rng.choice(
    len(s1_p),
    size=sample_size,
    replace=False
)

sample_s1 = (
    s1_p
    .iloc[sample_indices]
    .copy()
    .reset_index(drop=True)
)

sample_s1_ids = set(
    sample_s1["entity_id"]
)


print("\nControlled S1 sample:")
print(
    f"{len(sample_s1):,} Source-1 entities"
)


# ============================================================
# 12. BUILD GROUND-TRUTH MAP
# ============================================================

print("\nBuilding ground-truth map...")

valid_s2_ids = set(
    s2_p["entity_id"]
)

valid_s3_ids = set(
    s3_p["entity_id"]
)

source23_ids = valid_s2_ids | valid_s3_ids


gt_map = {}

gt_sample_match_count = 0
gt_invalid_count = 0


for row in gt.itertuples(index=False):

    s1_id = str(
        row.source1_entity_id
    )

    if s1_id not in sample_s1_ids:
        continue

    raw_matches = safe_text(
        row.matched_entity_ids
    ).strip()

    if not raw_matches:
        gt_map[s1_id] = set()
        continue

    matches = set(
        x.strip()
        for x in raw_matches.split(",")
        if x.strip()
    )

    valid_matches = (
        matches
        & source23_ids
    )

    invalid_matches = (
        matches
        - source23_ids
    )

    gt_invalid_count += len(
        invalid_matches
    )

    gt_map[s1_id] = valid_matches

    gt_sample_match_count += len(
        valid_matches
    )


# Make sure every sampled S1 has an entry.
for s1_id in sample_s1_ids:

    if s1_id not in gt_map:
        gt_map[s1_id] = set()


print(
    f"Sampled S1 entities in GT: "
    f"{len(gt_map):,}"
)

print(
    f"Ground-truth S2/S3 matches: "
    f"{gt_sample_match_count:,}"
)

print(
    f"Invalid GT references encountered: "
    f"{gt_invalid_count:,}"
)


# ============================================================
# 13. BLOCKING INDEX BUILDER
# ============================================================

def build_index(
    df,
    key_column,
    index_name
):
    """
    Build key -> entity IDs index.

    Empty keys are excluded.
    """

    index = defaultdict(list)

    keys = df[key_column].values
    ids = df["entity_id"].values

    for key, entity_id in zip(keys, ids):

        if not key:
            continue

        index[key].append(
            entity_id
        )

    print(
        f"{index_name:<35} "
        f"{len(index):,} unique keys"
    )

    return index


# ============================================================
# 14. BUILD S2/S3 BLOCK INDEXES
# ============================================================

print("\n")
print("=" * 80)
print("BUILDING BLOCKING INDEXES")
print("=" * 80)


index_specs = {
    "name_exact": "name_original",
    "name_casefold": "name_casefold",
    "name_sorted_tokens": "name_sorted_tokens",
    "address_exact": "address_original",
    "address_sorted_tokens": "address_sorted_tokens",
    "address_house_signature": "address_house_signature",
    "name_address_combined": "name_address_combined",
    "country_name_casefold": "country_name_casefold",
}


indexes = {}


index_start = perf_counter()


for block_name, column in index_specs.items():

    print(
        f"\n[{block_name}]"
    )

    indexes[
        block_name
    ] = {
        "S2": build_index(
            s2_p,
            column,
            f"S2 {block_name}"
        ),

        "S3": build_index(
            s3_p,
            column,
            f"S3 {block_name}"
        ),
    }


print(
    f"\nAll indexes built in "
    f"{perf_counter() - index_start:.2f} seconds."
)


# ============================================================
# 15. BLOCKING FUNCTION
# ============================================================

def get_block_key(
    row,
    block_name
):

    if block_name == "name_exact":
        return clean_key(
            row["name_original"]
        )

    if block_name == "name_casefold":
        return clean_key(
            row["name_casefold"]
        )

    if block_name == "name_sorted_tokens":
        return clean_key(
            row["name_sorted_tokens"]
        )

    if block_name == "address_exact":
        return clean_key(
            row["address_original"]
        )

    if block_name == "address_sorted_tokens":
        return clean_key(
            row["address_sorted_tokens"]
        )

    if block_name == "address_house_signature":
        return clean_key(
            row["address_house_signature"]
        )

    if block_name == "name_address_combined":
        name = clean_key(
            row["name_casefold"]
        )

        address = clean_key(
            row["address_casefold"]
        )

        if name is None or address is None:
            return None

        return (
            name
            + "||"
            + address
        )

    if block_name == "country_name_casefold":

        country = clean_key(
            row["country_key"]
        )

        name = clean_key(
            row["name_casefold"]
        )

        if country is None or name is None:
            return None

        return (
            country
            + "||"
            + name
        )

    raise ValueError(
        f"Unknown block: {block_name}"
    )


# ============================================================
# 16. CONTROLLED BLOCKING EXPERIMENT
# ============================================================

print("\n")
print("=" * 80)
print("RUNNING BLOCKING EXPERIMENT")
print("=" * 80)

experiment_start = perf_counter()


# Candidate sets:
#
# candidates[s1_id] = set(
#     "S2-...",
#     "S3-..."
# )
#
# Provenance:
#
# provenance[s1_id][candidate_id] = {
#     block_name,
#     ...
# }

candidates = defaultdict(set)

provenance = defaultdict(
    lambda: defaultdict(set)
)


processed = 0


for batch_start in range(
    0,
    len(sample_s1),
    BATCH_SIZE
):

    batch = sample_s1.iloc[
        batch_start:
        batch_start + BATCH_SIZE
    ]

    for row in batch.itertuples(
        index=False
    ):

        s1_id = row.entity_id

        for block_name in BLOCK_NAMES:

            key = get_block_key(
                row._asdict(),
                block_name
            )

            if key is None:
                continue

            # Retrieve from S2
            s2_matches = indexes[
                block_name
            ]["S2"].get(
                key,
                []
            )

            for candidate_id in s2_matches:

                candidates[s1_id].add(
                    candidate_id
                )

                if SAVE_PROVENANCE:
                    provenance[
                        s1_id
                    ][
                        candidate_id
                    ].add(
                        "S2:" + block_name
                    )

            # Retrieve from S3
            s3_matches = indexes[
                block_name
            ]["S3"].get(
                key,
                []
            )

            for candidate_id in s3_matches:

                candidates[s1_id].add(
                    candidate_id
                )

                if SAVE_PROVENANCE:
                    provenance[
                        s1_id
                    ][
                        candidate_id
                    ].add(
                        "S3:" + block_name
                    )

        processed += 1

    if (
        processed % 25_000 == 0
        or processed == len(sample_s1)
    ):

        elapsed = (
            perf_counter()
            - experiment_start
        )

        print(
            f"Processed "
            f"{processed:,}/"
            f"{len(sample_s1):,} "
            f"S1 | "
            f"{elapsed:.1f}s"
        )

    gc.collect()


experiment_runtime = (
    perf_counter()
    - experiment_start
)


print(
    "\nBlocking experiment completed."
)

print(
    f"Runtime: "
    f"{experiment_runtime:.2f} seconds"
)


# ============================================================
# 17. ENSURE ALL S1 HAVE CANDIDATE ENTRY
# ============================================================

for s1_id in sample_s1_ids:

    if s1_id not in candidates:
        candidates[s1_id] = set()


# ============================================================
# 18. CALCULATE BLOCKING METRICS
# ============================================================

candidate_counts = np.array(
    [
        len(candidates[s1_id])
        for s1_id in sample_s1_ids
    ],
    dtype=np.int64
)


total_candidate_pairs = int(
    candidate_counts.sum()
)


zero_candidate_count = int(
    (candidate_counts == 0).sum()
)


mean_candidates = float(
    candidate_counts.mean()
)


median_candidates = float(
    np.median(candidate_counts)
)


p95_candidates = float(
    np.percentile(
        candidate_counts,
        95
    )
)


max_candidates = int(
    candidate_counts.max()
)


# ============================================================
# 19. GROUND-TRUTH BLOCKING RECALL
# ============================================================

total_true_matches = 0
retrieved_true_matches = 0

s1_with_true_matches = 0
s1_with_all_true_matches = 0

s1_blocking_failures = []


for s1_id in sample_s1_ids:

    true_matches = gt_map.get(
        s1_id,
        set()
    )

    retrieved = candidates.get(
        s1_id,
        set()
    )

    total_true_matches += len(
        true_matches
    )

    retrieved_count = len(
        true_matches & retrieved
    )

    retrieved_true_matches += (
        retrieved_count
    )

    if true_matches:

        s1_with_true_matches += 1

        if true_matches.issubset(
            retrieved
        ):
            s1_with_all_true_matches += 1

        else:
            missing = (
                true_matches
                - retrieved
            )

            s1_blocking_failures.append(
                {
                    "source1_entity_id": s1_id,
                    "true_match_count": len(
                        true_matches
                    ),
                    "retrieved_true_count": retrieved_count,
                    "missing_true_count": len(
                        missing
                    ),
                    "missing_entity_ids": ",".join(
                        sorted(missing)
                    ),
                }
            )


if total_true_matches > 0:

    blocking_recall = (
        retrieved_true_matches
        / total_true_matches
    )

else:

    blocking_recall = np.nan


if s1_with_true_matches > 0:

    entity_level_recall = (
        s1_with_all_true_matches
        / s1_with_true_matches
    )

else:

    entity_level_recall = np.nan


# ============================================================
# 20. CANDIDATE REDUCTION RATIO
# ============================================================

total_possible_candidates = (
    len(s2_p)
    + len(s3_p)
)

if total_possible_candidates > 0:

    average_reduction_ratio = (
        1
        -
        (
            mean_candidates
            /
            total_possible_candidates
        )
    )

else:

    average_reduction_ratio = np.nan


# ============================================================
# 21. BLOCK-WISE DIAGNOSTICS
# ============================================================

print("\n")
print("=" * 80)
print("BLOCK-WISE DIAGNOSTICS")
print("=" * 80)


block_metrics = []


for block_name in BLOCK_NAMES:

    retrieved = 0
    true_total = 0

    s1_hit = 0

    for row in sample_s1.itertuples(
        index=False
    ):

        s1_id = row.entity_id

        true_matches = gt_map.get(
            s1_id,
            set()
        )

        if not true_matches:
            continue

        key = get_block_key(
            row._asdict(),
            block_name
        )

        if key is None:
            continue

        block_candidates = set()

        block_candidates.update(
            indexes[
                block_name
            ]["S2"].get(
                key,
                []
            )
        )

        block_candidates.update(
            indexes[
                block_name
            ]["S3"].get(
                key,
                []
            )
        )

        intersection = (
            true_matches
            & block_candidates
        )

        retrieved += len(
            intersection
        )

        true_total += len(
            true_matches
        )

        if intersection:
            s1_hit += 1

    recall = (
        retrieved / true_total
        if true_total
        else np.nan
    )

    entity_hit_rate = (
        s1_hit / s1_with_true_matches
        if s1_with_true_matches
        else np.nan
    )

    block_metrics.append(
        {
            "block": block_name,
            "true_matches_retrieved": retrieved,
            "total_true_matches": true_total,
            "pair_recall": recall,
            "s1_entities_hit": s1_hit,
            "s1_entities_with_true_matches": s1_with_true_matches,
            "entity_hit_rate": entity_hit_rate,
        }
    )


block_metrics_df = pd.DataFrame(
    block_metrics
)


# ============================================================
# 22. PRINT MAIN RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("M2 BLOCKING RESULTS")
print("=" * 80)

print(
    f"\nS1 entities evaluated:        "
    f"{len(sample_s1):,}"
)

print(
    f"Total true matches:           "
    f"{total_true_matches:,}"
)

print(
    f"Retrieved true matches:       "
    f"{retrieved_true_matches:,}"
)

print(
    f"Blocking recall:              "
    f"{blocking_recall * 100:.4f}%"
)

print(
    f"Entity-level complete recall: "
    f"{entity_level_recall * 100:.4f}%"
)

print(
    f"\nCandidate statistics:"
)

print(
    f"Total candidate pairs:        "
    f"{total_candidate_pairs:,}"
)

print(
    f"Mean candidates / S1:         "
    f"{mean_candidates:,.2f}"
)

print(
    f"Median candidates / S1:       "
    f"{median_candidates:,.2f}"
)

print(
    f"P95 candidates / S1:          "
    f"{p95_candidates:,.2f}"
)

print(
    f"Maximum candidates / S1:      "
    f"{max_candidates:,}"
)

print(
    f"Zero-candidate S1 entities:   "
    f"{zero_candidate_count:,}"
)

print(
    f"Zero-candidate rate:          "
    f"{zero_candidate_count / len(sample_s1) * 100:.4f}%"
)

print(
    f"\nRuntime:                      "
    f"{experiment_runtime:.2f} seconds"
)

print(
    f"Average candidate reduction: "
    f"{average_reduction_ratio * 100:.6f}%"
)


# ============================================================
# 23. BLOCK TABLE
# ============================================================

print("\n")
print("=" * 80)
print("INDIVIDUAL BLOCK PERFORMANCE")
print("=" * 80)

display(
    block_metrics_df.sort_values(
        "pair_recall",
        ascending=False
    ).reset_index(drop=True)
)


# ============================================================
# 24. BUILD CANDIDATE OUTPUT
# ============================================================

print("\nCreating candidate output...")

candidate_rows = []


for s1_id in sorted(
    sample_s1_ids
):

    candidate_ids = sorted(
        candidates[s1_id]
    )

    candidate_rows.append(
        {
            "source1_entity_id": s1_id,
            "candidate_entity_ids": ",".join(
                candidate_ids
            ),
            "candidate_count": len(
                candidate_ids
            ),
        }
    )


candidate_df = pd.DataFrame(
    candidate_rows
)


# ============================================================
# 25. SAVE CANDIDATE OUTPUT
# ============================================================

candidate_path = (
    OUTPUT_DIR
    /
    "candidate_pairs_100k.tsv"
)


candidate_df[
    [
        "source1_entity_id",
        "candidate_entity_ids",
    ]
].to_csv(
    candidate_path,
    sep="\t",
    index=False
)


print(
    f"Saved candidate pairs:\n"
    f"{candidate_path}"
)


# ============================================================
# 26. SAVE DETAILED METRICS
# ============================================================

metrics = {
    "experiment": "M2_STAGE1_BLOCKING_100K",
    "random_seed": RANDOM_SEED,
    "s1_sample_size": len(sample_s1),
    "batch_size": BATCH_SIZE,

    "total_true_matches": total_true_matches,
    "retrieved_true_matches": retrieved_true_matches,

    "blocking_recall": blocking_recall,
    "entity_level_complete_recall": entity_level_recall,

    "total_candidate_pairs": total_candidate_pairs,
    "mean_candidates": mean_candidates,
    "median_candidates": median_candidates,
    "p95_candidates": p95_candidates,
    "max_candidates": max_candidates,

    "zero_candidate_count": zero_candidate_count,

    "zero_candidate_rate": (
        zero_candidate_count
        / len(sample_s1)
    ),

    "average_candidate_reduction_ratio":
        average_reduction_ratio,

    "runtime_seconds":
        experiment_runtime,

    "total_s2_rows":
        len(s2_p),

    "total_s3_rows":
        len(s3_p),

    "total_possible_candidates":
        total_possible_candidates,
}


metrics_path = (
    OUTPUT_DIR
    /
    "blocking_metrics_100k.json"
)


with open(
    metrics_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metrics,
        f,
        indent=2
    )


block_metrics_path = (
    OUTPUT_DIR
    /
    "block_level_metrics_100k.csv"
)


block_metrics_df.to_csv(
    block_metrics_path,
    index=False
)


print(
    f"\nSaved metrics:\n"
    f"{metrics_path}"
)

print(
    f"Saved block metrics:\n"
    f"{block_metrics_path}"
)


# ============================================================
# 27. SAVE BLOCKING FAILURES
# ============================================================

failure_df = pd.DataFrame(
    s1_blocking_failures
)


failure_path = (
    OUTPUT_DIR
    /
    "blocking_failures_100k.csv"
)


failure_df.to_csv(
    failure_path,
    index=False
)


print(
    f"Saved blocking failures:\n"
    f"{failure_path}"
)


# ============================================================
# 28. SAMPLE CANDIDATE INSPECTION
# ============================================================

print("\n")
print("=" * 80)
print("SAMPLE CANDIDATE INSPECTION")
print("=" * 80)


inspection_ids = list(
    sample_s1_ids
)[:10]


inspection_rows = []


for s1_id in inspection_ids:

    true_matches = gt_map.get(
        s1_id,
        set()
    )

    candidate_set = candidates.get(
        s1_id,
        set()
    )

    inspection_rows.append(
        {
            "source1_entity_id":
                s1_id,

            "true_match_count":
                len(true_matches),

            "candidate_count":
                len(candidate_set),

            "true_matches_retrieved":
                len(
                    true_matches
                    &
                    candidate_set
                ),

            "all_true_matches_retrieved":
                true_matches.issubset(
                    candidate_set
                ),
        }
    )


inspection_df = pd.DataFrame(
    inspection_rows
)

display(
    inspection_df
)


# ============================================================
# 29. SAVE REPORT
# ============================================================

report_path = (
    REPORT_DIR
    /
    "M2_BLOCKING_100K_RESULTS.md"
)


report_lines = []

report_lines.append(
    "# M2 — Blocking & Candidate Retrieval\n"
)

report_lines.append(
    "\n## Stage 1 — Controlled 100K Experiment\n"
)

report_lines.append(
    "\n### Configuration\n"
)

report_lines.append(
    f"- Random seed: `{RANDOM_SEED}`\n"
)

report_lines.append(
    f"- Source 1 sample: **{len(sample_s1):,}**\n"
)

report_lines.append(
    f"- Batch size: **{BATCH_SIZE:,}**\n"
)

report_lines.append(
    "\n### Blocking Strategies\n"
)

for block_name in BLOCK_NAMES:

    report_lines.append(
        f"- `{block_name}`\n"
    )


report_lines.append(
    "\n### Results\n"
)

report_lines.append(
    f"- Total ground-truth matches: "
    f"**{total_true_matches:,}**\n"
)

report_lines.append(
    f"- Retrieved ground-truth matches: "
    f"**{retrieved_true_matches:,}**\n"
)

report_lines.append(
    f"- Pair-level blocking recall: "
    f"**{blocking_recall * 100:.4f}%**\n"
)

report_lines.append(
    f"- Entity-level complete recall: "
    f"**{entity_level_recall * 100:.4f}%**\n"
)

report_lines.append(
    f"- Total candidate pairs: "
    f"**{total_candidate_pairs:,}**\n"
)

report_lines.append(
    f"- Mean candidates per S1: "
    f"**{mean_candidates:,.2f}**\n"
)

report_lines.append(
    f"- Median candidates per S1: "
    f"**{median_candidates:,.2f}**\n"
)

report_lines.append(
    f"- P95 candidates per S1: "
    f"**{p95_candidates:,.2f}**\n"
)

report_lines.append(
    f"- Maximum candidates per S1: "
    f"**{max_candidates:,}**\n"
)

report_lines.append(
    f"- Zero-candidate S1 entities: "
    f"**{zero_candidate_count:,}**\n"
)

report_lines.append(
    f"- Runtime: "
    f"**{experiment_runtime:.2f} seconds**\n"
)

report_lines.append(
    "\n### Block-level results\n\n"
)

report_lines.append(
    block_metrics_df.to_markdown(
        index=False
    )
)

report_lines.append(
    "\n\n### Important interpretation\n\n"
)

report_lines.append(
    "This experiment measures candidate-generation recall "
    "against the training ground truth. A blocking strategy "
    "is not considered sufficient merely because it produces "
    "a small candidate set; all true matches must remain "
    "available for the downstream pair model.\n"
)

report_lines.append(
    "\nBusiness-name `digits` is intentionally not used as "
    "a standalone blocking key because M1 validation showed "
    "that its production recovery was only 1.4154% and that "
    "empty numeric representations must not be treated as "
    "matching evidence. It remains available as an auxiliary "
    "feature for later pair scoring.\n"
)

report_lines.append(
    "\nThe negative-collision analysis from M1 was sampled, "
    "so blocking collision behavior should also be examined "
    "empirically rather than interpreted as an exhaustive "
    "full-dataset collision probability.\n"
)


report_path.write_text(
    "\n".join(report_lines),
    encoding="utf-8"
)


print(
    f"\nReport saved:\n"
    f"{report_path}"
)


# ============================================================
# 30. FINAL STATUS
# ============================================================

print("\n")
print("=" * 80)
print("M2 STAGE 1 COMPLETED")
print("=" * 80)

print(
    "\nNext stage should NOT yet be the pair model."
)

print(
    "First review:"
)

print(
    "1. Overall blocking recall"
)

print(
    "2. Individual block recall"
)

print(
    "3. Candidate volume"
)

print(
    "4. P95 / maximum candidate explosion"
)

print(
    "5. Blocking failures"
)

print(
    "6. Whether country-aware blocking should be retained"
)

print(
    "\nOnly after high-recall blocking is established "
    "should we proceed to advanced retrieval / fuzzy blocking."
)

print("\nOutput files:")

print(
    candidate_path
)

print(
    metrics_path
)

print(
    block_metrics_path
)

print(
    failure_path
)

print(
    report_path
)

print("\n" + "=" * 80)

M2 — BLOCKING & CANDIDATE RETRIEVAL
STAGE 1 — CONTROLLED 100K EXPERIMENT

Project root:
C:\Amazon ML Challenge\student_resource

Training directory:
C:\Amazon ML Challenge\student_resource\dataset\train

Output directory:
C:\Amazon ML Challenge\student_resource\m2_work\outputs\blocking_100k

Batch size:
5,000

S1 sample size:
100,000

Checking files...
train_source1.tsv              OK
train_source2.tsv              OK
train_source3.tsv              OK
train_ground_truth.tsv         OK

Loading Source 1...
Source 1: (2206821, 4)

Loading Source 2...
Source 2: (5034616, 4)

Loading Source 3...
Source 3: (5285603, 4)

Loading ground truth...
Ground truth: (2206821, 2)

ID validation...
S1: 2,206,821 rows | 2,206,821 unique IDs | 0 duplicate IDs
S2: 5,034,616 rows | 5,034,616 unique IDs | 0 duplicate IDs
S3: 5,285,603 rows | 5,285,603 unique IDs | 0 duplicate IDs

Preparing normalized representations...

Normalizing Source 1...

Normalizing Source 2...

Normalizing Source 3...

Representa

KeyboardInterrupt: 

In [ ]:
# ============================================================
# M2.2 — RARE TOKEN + COUNTRY-AWARE BLOCKING
# Amazon ML Challenge 2026
#
# SINGLE-CELL VERSION
#
# M2.1 candidate schema:
#   source1_entity_id
#   candidate_entity_ids
#
# M2.2 adds:
#   - Country-aware rare name tokens
#   - Country-aware rare address tokens
#   - DF thresholds: 2,5,10,25,50
#   - Union with M2.1
#   - Incremental recall
#   - Entity-complete recall
#   - Candidate-constrained macro F0.5 oracle
#   - Failure analysis
#
# IMPORTANT:
#   No candidate cap is applied.
#   Recall is prioritized over candidate reduction.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
import re
import unicodedata
import time
import gc
import traceback


# ============================================================
# 0. CONFIGURATION
# ============================================================

SEED = 42
S1_SAMPLE_SIZE = 100_000
CHUNK_SIZE = 100_000

DF_THRESHOLDS = [2, 5, 10, 25, 50]
MAX_DF = max(DF_THRESHOLDS)

PROJECT_ROOT = Path.cwd().parent.parent

if not (PROJECT_ROOT / "dataset").exists():
    PROJECT_ROOT = Path.cwd()

TRAIN_DIR = PROJECT_ROOT / "dataset" / "train"

S1_PATH = TRAIN_DIR / "train_source1.tsv"
S2_PATH = TRAIN_DIR / "train_source2.tsv"
S3_PATH = TRAIN_DIR / "train_source3.tsv"
GT_PATH = TRAIN_DIR / "train_ground_truth.tsv"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "m2_work"
    / "outputs"
    / "blocking_100k_m2_2"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "m2_work"
    / "reports"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("M2.2 — RARE TOKEN + COUNTRY-AWARE BLOCKING")
print("=" * 100)
print("PROJECT ROOT :", PROJECT_ROOT)
print("TRAIN DIR    :", TRAIN_DIR)
print("OUTPUT DIR   :", OUTPUT_DIR)


# ============================================================
# 1. VERIFY FILES
# ============================================================

for path in [
    S1_PATH,
    S2_PATH,
    S3_PATH,
    GT_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"\nRequired file not found:\n{path}"
        )

    print("FOUND:", path)


# ============================================================
# 2. NORMALIZATION FUNCTIONS
# ============================================================

def normalize_unicode(text):
    if text is None:
        return ""
    return unicodedata.normalize("NFKC", str(text))


def normalize_casefold(text):
    return normalize_unicode(text).casefold()


def normalize_alphanumeric(text):
    text = normalize_casefold(text)

    text = re.sub(
        r"[^\w]+",
        " ",
        text,
        flags=re.UNICODE
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokenize(text):
    text = normalize_alphanumeric(text)

    if not text:
        return []

    return [
        x for x in text.split()
        if x
    ]


def unique_tokens(text):
    return set(tokenize(text))


def country_key(text):
    text = normalize_casefold(text)

    text = re.sub(
        r"[^\w]+",
        " ",
        text,
        flags=re.UNICODE
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def safe_text(value):
    if value is None:
        return ""
    return str(value).strip()


# ============================================================
# 3. LOAD S1 IDS AND SAMPLE 100K
# ============================================================

print("\n" + "=" * 100)
print("STEP 1 — 100K S1 SAMPLE")
print("=" * 100)

start = time.time()

s1_ids = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=["entity_id"],
    dtype=str,
    keep_default_na=False,
)

print(
    "Total S1:",
    f"{len(s1_ids):,}"
)

rng = np.random.default_rng(SEED)

sample_indices = rng.choice(
    len(s1_ids),
    size=min(
        S1_SAMPLE_SIZE,
        len(s1_ids)
    ),
    replace=False,
)

sample_s1_ids = set(
    s1_ids.iloc[
        sample_indices
    ]["entity_id"].tolist()
)

del s1_ids
gc.collect()

print(
    "Sample S1:",
    f"{len(sample_s1_ids):,}"
)

print(
    "Time:",
    round(time.time() - start, 2),
    "seconds"
)


# ============================================================
# 4. LOAD SAMPLED S1 RECORDS
# ============================================================

print("\n" + "=" * 100)
print("STEP 2 — LOAD SAMPLED S1 RECORDS")
print("=" * 100)

start = time.time()

S1_COLUMNS = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

s1_parts = []

for chunk in pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=S1_COLUMNS,
    dtype=str,
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
):

    mask = chunk[
        "entity_id"
    ].isin(sample_s1_ids)

    if mask.any():

        s1_parts.append(
            chunk.loc[
                mask,
                S1_COLUMNS
            ].copy()
        )

s1_sample = pd.concat(
    s1_parts,
    ignore_index=True
)

del s1_parts
gc.collect()

if len(s1_sample) != len(sample_s1_ids):
    raise RuntimeError(
        "S1 sample integrity check failed."
    )

print(
    "Loaded:",
    f"{len(s1_sample):,}"
)

print(
    "Time:",
    round(time.time() - start, 2),
    "seconds"
)


# ============================================================
# 5. LOAD GROUND TRUTH
# ============================================================

print("\n" + "=" * 100)
print("STEP 3 — LOAD GROUND TRUTH")
print("=" * 100)

GT_COLUMNS = [
    "source1_entity_id",
    "matched_entity_ids",
]

gt_parts = []

for chunk in pd.read_csv(
    GT_PATH,
    sep="\t",
    usecols=GT_COLUMNS,
    dtype=str,
    keep_default_na=False,
    chunksize=CHUNK_SIZE,
):

    mask = chunk[
        "source1_entity_id"
    ].isin(sample_s1_ids)

    if mask.any():

        gt_parts.append(
            chunk.loc[
                mask,
                GT_COLUMNS
            ].copy()
        )

gt_sample = pd.concat(
    gt_parts,
    ignore_index=True
)

del gt_parts
gc.collect()


gt_lookup = {}

for row in gt_sample.itertuples(
    index=False
):

    s1_id = row.source1_entity_id

    matched = safe_text(
        row.matched_entity_ids
    )

    if matched:

        targets = {
            x.strip()
            for x in matched.split(",")
            if x.strip()
        }

    else:

        targets = set()

    gt_lookup[s1_id] = targets


for s1_id in sample_s1_ids:
    gt_lookup.setdefault(
        s1_id,
        set()
    )


gt_pairs = set()

for s1_id, targets in gt_lookup.items():

    for target_id in targets:

        gt_pairs.add(
            (
                s1_id,
                target_id
            )
        )


s1_with_true_matches = sum(
    bool(x)
    for x in gt_lookup.values()
)

zero_match_s1 = (
    len(gt_lookup)
    - s1_with_true_matches
)

print(
    "GT S1 rows:",
    f"{len(gt_sample):,}"
)

print(
    "S1 entities:",
    f"{len(gt_lookup):,}"
)

print(
    "S1 with true matches:",
    f"{s1_with_true_matches:,}"
)

print(
    "Zero-match S1:",
    f"{zero_match_s1:,}"
)

print(
    "True pairs:",
    f"{len(gt_pairs):,}"
)


# ============================================================
# 6. TOKEN FREQUENCY PASS
# ============================================================

print("\n" + "=" * 100)
print("STEP 4 — TOKEN FREQUENCY PASS")
print("=" * 100)


def count_tokens_in_source(path):

    name_counts = Counter()
    address_counts = Counter()

    rows = 0
    start = time.time()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=[
            "business_name",
            "business_address",
            "country"
        ],
        dtype=str,
        keep_default_na=False,
        chunksize=CHUNK_SIZE,
    ):

        for (
            name,
            address,
            country
        ) in zip(
            chunk["business_name"],
            chunk["business_address"],
            chunk["country"]
        ):

            c = country_key(country)

            if not c:
                c = "__EMPTY_COUNTRY__"

            for token in unique_tokens(name):

                name_counts[
                    (c, token)
                ] += 1

            for token in unique_tokens(address):

                address_counts[
                    (c, token)
                ] += 1

        rows += len(chunk)

        if rows % 1_000_000 == 0:

            print(
                path.name,
                "processed:",
                f"{rows:,}"
            )

    print(
        path.name,
        "completed:",
        f"{rows:,}",
        "rows",
        round(time.time() - start, 2),
        "sec"
    )

    return (
        name_counts,
        address_counts
    )


print("\nS2 token frequency...")

(
    s2_name_counts,
    s2_address_counts
) = count_tokens_in_source(
    S2_PATH
)

print("\nS3 token frequency...")

(
    s3_name_counts,
    s3_address_counts
) = count_tokens_in_source(
    S3_PATH
)


print(
    "\nS2 name keys:",
    f"{len(s2_name_counts):,}"
)

print(
    "S2 address keys:",
    f"{len(s2_address_counts):,}"
)

print(
    "S3 name keys:",
    f"{len(s3_name_counts):,}"
)

print(
    "S3 address keys:",
    f"{len(s3_address_counts):,}"
)


# ============================================================
# 7. BUILD RARE TOKEN INDEX
# ============================================================

print("\n" + "=" * 100)
print(
    f"STEP 5 — BUILD RARE TOKEN INDEX (DF <= {MAX_DF})"
)
print("=" * 100)


def build_token_index(
    path,
    name_counts,
    address_counts,
    max_df
):

    name_index = defaultdict(list)
    address_index = defaultdict(list)

    rows = 0
    start = time.time()

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=[
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ],
        dtype=str,
        keep_default_na=False,
        chunksize=CHUNK_SIZE,
    ):

        for (
            entity_id,
            name,
            address,
            country
        ) in zip(
            chunk["entity_id"],
            chunk["business_name"],
            chunk["business_address"],
            chunk["country"]
        ):

            c = country_key(country)

            if not c:
                c = "__EMPTY_COUNTRY__"

            # -----------------------------
            # NAME
            # -----------------------------

            for token in unique_tokens(name):

                key = (
                    c,
                    token
                )

                if (
                    name_counts.get(
                        key,
                        0
                    )
                    <= max_df
                ):

                    name_index[
                        key
                    ].append(
                        entity_id
                    )

            # -----------------------------
            # ADDRESS
            # -----------------------------

            for token in unique_tokens(address):

                key = (
                    c,
                    token
                )

                if (
                    address_counts.get(
                        key,
                        0
                    )
                    <= max_df
                ):

                    address_index[
                        key
                    ].append(
                        entity_id
                    )

        rows += len(chunk)

        if rows % 1_000_000 == 0:

            print(
                path.name,
                "indexed:",
                f"{rows:,}"
            )

    print(
        path.name,
        "completed:",
        f"{rows:,}",
        "rows",
        round(time.time() - start, 2),
        "sec"
    )

    return (
        name_index,
        address_index
    )


print("\nBuilding S2 index...")

(
    s2_name_index,
    s2_address_index
) = build_token_index(
    S2_PATH,
    s2_name_counts,
    s2_address_counts,
    MAX_DF
)

print("\nBuilding S3 index...")

(
    s3_name_index,
    s3_address_index
) = build_token_index(
    S3_PATH,
    s3_name_counts,
    s3_address_counts,
    MAX_DF
)

print(
    "\nS2 name index:",
    f"{len(s2_name_index):,}"
)

print(
    "S2 address index:",
    f"{len(s2_address_index):,}"
)

print(
    "S3 name index:",
    f"{len(s3_name_index):,}"
)

print(
    "S3 address index:",
    f"{len(s3_address_index):,}"
)


# ============================================================
# 8. LOAD M2.1 CANDIDATES
# ============================================================

print("\n" + "=" * 100)
print("STEP 6 — LOAD M2.1 CANDIDATES")
print("=" * 100)

M21_CANDIDATE_PATH = (
    PROJECT_ROOT
    / "m2_work"
    / "outputs"
    / "blocking_100k"
    / "candidate_pairs_100k.tsv"
)

if not M21_CANDIDATE_PATH.exists():

    alternatives = list(
        (
            PROJECT_ROOT
            / "m2_work"
            / "outputs"
        ).glob(
            "**/candidate_pairs_100k.tsv"
        )
    )

    if alternatives:
        M21_CANDIDATE_PATH = alternatives[0]

    else:
        raise FileNotFoundError(
            "\nM2.1 candidate_pairs_100k.tsv "
            "could not be found."
        )

print(
    "M2.1 file:",
    M21_CANDIDATE_PATH
)

m21_candidates = pd.read_csv(
    M21_CANDIDATE_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print(
    "Columns:",
    m21_candidates.columns.tolist()
)


# ============================================================
# 9. EXPAND M2.1 candidate_entity_ids
# ============================================================

print("\n" + "=" * 100)
print("STEP 7 — EXPAND M2.1 CANDIDATE LISTS")
print("=" * 100)

if "source1_entity_id" not in m21_candidates.columns:
    raise ValueError(
        "source1_entity_id not found."
    )

if "candidate_entity_ids" not in m21_candidates.columns:
    raise ValueError(
        "candidate_entity_ids not found."
    )


m21_pair_set = set()

for row in m21_candidates.itertuples(
    index=False
):

    s1_id = safe_text(
        row.source1_entity_id
    )

    candidate_string = safe_text(
        row.candidate_entity_ids
    )

    if not s1_id:
        continue

    if not candidate_string:
        continue

    for target_id in candidate_string.split(","):

        target_id = target_id.strip()

        if not target_id:
            continue

        if (
            target_id.startswith("S2-")
            or target_id.startswith("S3-")
        ):

            m21_pair_set.add(
                (
                    s1_id,
                    target_id
                )
            )


print(
    "M2.1 unique candidate pairs:",
    f"{len(m21_pair_set):,}"
)


# ============================================================
# 10. M2.1 BASELINE
# ============================================================

m21_true = (
    m21_pair_set
    & gt_pairs
)

m21_recall = (
    len(m21_true)
    / len(gt_pairs)
    if gt_pairs
    else 0.0
)

print(
    "\nM2.1 true pairs retrieved:",
    f"{len(m21_true):,}"
)

print(
    "M2.1 pair recall:",
    f"{m21_recall:.6%}"
)


# ============================================================
# 11. RARE TOKEN QUERY
# ============================================================

def query_rare_candidates(
    row,
    max_df
):

    country = country_key(
        row["country"]
    )

    if not country:
        country = "__EMPTY_COUNTRY__"

    candidates = set()

    provenance = defaultdict(set)

    # ========================================================
    # NAME
    # ========================================================

    for token in unique_tokens(
        row["business_name"]
    ):

        key = (
            country,
            token
        )

        if (
            s2_name_counts.get(
                key,
                0
            )
            <= max_df
        ):

            for target in s2_name_index.get(
                key,
                []
            ):

                candidates.add(target)

                provenance[target].add(
                    "rare_name"
                )

        if (
            s3_name_counts.get(
                key,
                0
            )
            <= max_df
        ):

            for target in s3_name_index.get(
                key,
                []
            ):

                candidates.add(target)

                provenance[target].add(
                    "rare_name"
                )

    # ========================================================
    # ADDRESS
    # ========================================================

    for token in unique_tokens(
        row["business_address"]
    ):

        key = (
            country,
            token
        )

        if (
            s2_address_counts.get(
                key,
                0
            )
            <= max_df
        ):

            for target in s2_address_index.get(
                key,
                []
            ):

                candidates.add(target)

                provenance[target].add(
                    "rare_address"
                )

        if (
            s3_address_counts.get(
                key,
                0
            )
            <= max_df
        ):

            for target in s3_address_index.get(
                key,
                []
            ):

                candidates.add(target)

                provenance[target].add(
                    "rare_address"
                )

    return (
        candidates,
        provenance
    )


# ============================================================
# 12. RUN M2.2 FOR ALL THRESHOLDS
# ============================================================

print("\n" + "=" * 100)
print("STEP 8 — M2.2 THRESHOLD EXPERIMENT")
print("=" * 100)

s1_records = s1_sample.to_dict(
    "records"
)

threshold_results = []
threshold_candidate_sets = {}
threshold_block_pairs = {}

overall_start = time.time()


for threshold in DF_THRESHOLDS:

    print("\n")
    print("=" * 100)
    print(
        f"DF THRESHOLD = {threshold}"
    )
    print("=" * 100)

    threshold_start = time.time()

    candidate_pairs = set()
    rare_name_pairs = set()
    rare_address_pairs = set()

    for idx, row in enumerate(
        s1_records,
        start=1
    ):

        s1_id = row[
            "entity_id"
        ]

        (
            candidates,
            provenance
        ) = query_rare_candidates(
            row,
            threshold
        )

        for target_id in candidates:

            pair = (
                s1_id,
                target_id
            )

            candidate_pairs.add(
                pair
            )

            blocks = provenance[
                target_id
            ]

            if "rare_name" in blocks:

                rare_name_pairs.add(
                    pair
                )

            if "rare_address" in blocks:

                rare_address_pairs.add(
                    pair
                )

        if idx % 10_000 == 0:

            print(
                f"{idx:,}/{len(s1_records):,} "
                f"S1 processed | "
                f"{len(candidate_pairs):,} "
                f"rare candidates"
            )

    threshold_candidate_sets[
        threshold
    ] = candidate_pairs

    threshold_block_pairs[
        threshold
    ] = {
        "rare_name":
            rare_name_pairs,

        "rare_address":
            rare_address_pairs
    }

    # ========================================================
    # RARE ONLY
    # ========================================================

    rare_true = (
        candidate_pairs
        & gt_pairs
    )

    rare_recall = (
        len(rare_true)
        / len(gt_pairs)
    )

    # ========================================================
    # UNION WITH M2.1
    # ========================================================

    union_pairs = (
        m21_pair_set
        | candidate_pairs
    )

    union_true = (
        union_pairs
        & gt_pairs
    )

    new_true = (
        union_true
        - m21_true
    )

    union_recall = (
        len(union_true)
        / len(gt_pairs)
    )

    incremental_gain = (
        len(new_true)
        / len(gt_pairs)
    )

    # ========================================================
    # ENTITY COMPLETE RECALL
    # ========================================================

    candidate_by_s1 = defaultdict(set)

    for (
        sid,
        target
    ) in union_pairs:

        candidate_by_s1[
            sid
        ].add(target)

    complete_entities = 0
    positive_complete = 0

    for (
        s1_id,
        true_targets
    ) in gt_lookup.items():

        candidates = (
            candidate_by_s1.get(
                s1_id,
                set()
            )
        )

        if not true_targets:

            if not candidates:
                complete_entities += 1

        else:

            if true_targets.issubset(
                candidates
            ):

                complete_entities += 1
                positive_complete += 1

    entity_complete_recall = (
        complete_entities
        / len(gt_lookup)
    )

    positive_entity_complete_recall = (
        positive_complete
        / s1_with_true_matches
        if s1_with_true_matches
        else 0.0
    )

    # ========================================================
    # CANDIDATE DISTRIBUTION
    # ========================================================

    counts = np.fromiter(
        (
            len(
                candidate_by_s1.get(
                    s1_id,
                    set()
                )
            )
            for s1_id in sample_s1_ids
        ),
        dtype=np.int64
    )

    runtime = (
        time.time()
        - threshold_start
    )

    threshold_results.append(
        {
            "df_threshold":
                threshold,

            "rare_candidate_pairs":
                len(candidate_pairs),

            "rare_pair_recall":
                rare_recall,

            "union_candidate_pairs":
                len(union_pairs),

            "new_true_pairs_recovered":
                len(new_true),

            "union_true_pairs":
                len(union_true),

            "union_pair_recall":
                union_recall,

            "incremental_recall_gain":
                incremental_gain,

            "entity_complete_recall":
                entity_complete_recall,

            "positive_entity_complete_recall":
                positive_entity_complete_recall,

            "mean_candidates":
                float(counts.mean()),

            "median_candidates":
                float(np.median(counts)),

            "p95_candidates":
                float(
                    np.percentile(
                        counts,
                        95
                    )
                ),

            "p99_candidates":
                float(
                    np.percentile(
                        counts,
                        99
                    )
                ),

            "max_candidates":
                int(counts.max()),

            "zero_candidate_s1":
                int(
                    (counts == 0).sum()
                ),

            "runtime_seconds":
                runtime
        }
    )

    print("\nRESULT")
    print(
        "Rare-only recall       :",
        f"{rare_recall:.6%}"
    )
    print(
        "New true pairs         :",
        f"{len(new_true):,}"
    )
    print(
        "Union recall           :",
        f"{union_recall:.6%}"
    )
    print(
        "Entity complete recall :",
        f"{entity_complete_recall:.6%}"
    )
    print(
        "Union candidates       :",
        f"{len(union_pairs):,}"
    )
    print(
        "Mean candidates/S1     :",
        f"{counts.mean():.2f}"
    )
    print(
        "P95 candidates/S1      :",
        f"{np.percentile(counts, 95):.2f}"
    )
    print(
        "P99 candidates/S1      :",
        f"{np.percentile(counts, 99):.2f}"
    )
    print(
        "Runtime                :",
        f"{runtime:.2f}s"
    )


threshold_results_df = pd.DataFrame(
    threshold_results
)


# ============================================================
# 13. BLOCK LEVEL RESULTS
# ============================================================

print("\n" + "=" * 100)
print("STEP 9 — BLOCK LEVEL RESULTS")
print("=" * 100)

block_rows = []

for threshold in DF_THRESHOLDS:

    for (
        block_name,
        block_pairs
    ) in threshold_block_pairs[
        threshold
    ].items():

        block_true = (
            block_pairs
            & gt_pairs
        )

        block_rows.append(
            {
                "df_threshold":
                    threshold,

                "block":
                    block_name,

                "candidate_pairs":
                    len(block_pairs),

                "true_pairs_retrieved":
                    len(block_true),

                "pair_recall":
                    (
                        len(block_true)
                        / len(gt_pairs)
                    )
            }
        )

block_results_df = pd.DataFrame(
    block_rows
)

print(
    block_results_df.to_string(
        index=False
    )
)


# ============================================================
# 14. INCREMENTAL M2.1 ANALYSIS
# ============================================================

print("\n" + "=" * 100)
print("STEP 10 — INCREMENTAL GAIN OVER M2.1")
print("=" * 100)

incremental_rows = []

for threshold in DF_THRESHOLDS:

    rare_pairs = (
        threshold_candidate_sets[
            threshold
        ]
    )

    union_pairs = (
        m21_pair_set
        | rare_pairs
    )

    union_true = (
        union_pairs
        & gt_pairs
    )

    new_true = (
        union_true
        - m21_true
    )

    incremental_rows.append(
        {
            "df_threshold":
                threshold,

            "M2_1_true_pairs":
                len(m21_true),

            "rare_block_true_pairs":
                len(
                    rare_pairs
                    & gt_pairs
                ),

            "new_true_pairs_recovered":
                len(new_true),

            "union_true_pairs":
                len(union_true),

            "union_pair_recall":
                (
                    len(union_true)
                    / len(gt_pairs)
                ),

            "incremental_recall_gain":
                (
                    len(new_true)
                    / len(gt_pairs)
                ),

            "union_candidate_pairs":
                len(union_pairs)
        }
    )

incremental_results_df = pd.DataFrame(
    incremental_rows
)

print(
    incremental_results_df.to_string(
        index=False
    )
)


# ============================================================
# 15. ORACLE MACRO F0.5
# ============================================================

print("\n" + "=" * 100)
print("STEP 11 — CANDIDATE-CONSTRAINED ORACLE F0.5")
print("=" * 100)


def oracle_macro_f05(
    candidate_pairs,
    gt_lookup
):

    by_s1 = defaultdict(set)

    for (
        s1_id,
        target_id
    ) in candidate_pairs:

        by_s1[
            s1_id
        ].add(target_id)

    scores = []

    for (
        s1_id,
        true_targets
    ) in gt_lookup.items():

        true_count = len(
            true_targets
        )

        # ----------------------------------------------------
        # ZERO-MATCH ENTITY
        # ----------------------------------------------------

        if true_count == 0:

            score = 1.0

        else:

            candidates = (
                by_s1.get(
                    s1_id,
                    set()
                )
            )

            recovered = len(
                true_targets
                & candidates
            )

            if recovered == 0:

                score = 0.0

            else:

                precision = 1.0

                recall = (
                    recovered
                    / true_count
                )

                score = (
                    1.25
                    * precision
                    * recall
                    / (
                        0.25
                        * precision
                        + recall
                    )
                )

        scores.append(score)

    return float(
        np.mean(scores)
    )


oracle_rows = []

for threshold in DF_THRESHOLDS:

    rare_pairs = (
        threshold_candidate_sets[
            threshold
        ]
    )

    union_pairs = (
        m21_pair_set
        | rare_pairs
    )

    union_true = (
        union_pairs
        & gt_pairs
    )

    oracle_score = oracle_macro_f05(
        union_pairs,
        gt_lookup
    )

    oracle_rows.append(
        {
            "df_threshold":
                threshold,

            "candidate_pairs":
                len(union_pairs),

            "pair_recall":
                (
                    len(union_true)
                    / len(gt_pairs)
                ),

            "oracle_macro_f05":
                oracle_score
        }
    )

oracle_df = pd.DataFrame(
    oracle_rows
)

print(
    oracle_df.to_string(
        index=False
    )
)


# ============================================================
# 16. SELECT BEST CONFIGURATION
# ============================================================

print("\n" + "=" * 100)
print("STEP 12 — SELECT BEST M2.2 CONFIGURATION")
print("=" * 100)

best_row = (
    oracle_df
    .sort_values(
        [
            "oracle_macro_f05",
            "pair_recall"
        ],
        ascending=[
            False,
            False
        ]
    )
    .iloc[0]
)

BEST_THRESHOLD = int(
    best_row["df_threshold"]
)

best_rare_pairs = (
    threshold_candidate_sets[
        BEST_THRESHOLD
    ]
)

best_union_pairs = (
    m21_pair_set
    | best_rare_pairs
)

print(
    "Best DF threshold:",
    BEST_THRESHOLD
)

print(
    "Best candidate pairs:",
    f"{len(best_union_pairs):,}"
)

print(
    "Best pair recall:",
    f"{best_row['pair_recall']:.6%}"
)

print(
    "Best oracle macro F0.5:",
    f"{best_row['oracle_macro_f05']:.6%}"
)


# ============================================================
# 17. FAILURE ANALYSIS
# ============================================================

print("\n" + "=" * 100)
print("STEP 13 — FAILURE ANALYSIS")
print("=" * 100)

candidate_by_s1_best = defaultdict(set)

for (
    s1_id,
    target_id
) in best_union_pairs:

    candidate_by_s1_best[
        s1_id
    ].add(target_id)


failures = []

for row in s1_records:

    s1_id = row[
        "entity_id"
    ]

    true_targets = gt_lookup[
        s1_id
    ]

    if not true_targets:
        continue

    candidates = (
        candidate_by_s1_best.get(
            s1_id,
            set()
        )
    )

    missing = (
        true_targets
        - candidates
    )

    if missing:

        failures.append(
            {
                "s1_id":
                    s1_id,

                "business_name":
                    row[
                        "business_name"
                    ],

                "business_address":
                    row[
                        "business_address"
                    ],

                "country":
                    row[
                        "country"
                    ],

                "true_match_count":
                    len(true_targets),

                "candidate_count":
                    len(candidates),

                "missing_true_match_count":
                    len(missing),

                "missing_true_matches":
                    ",".join(
                        sorted(
                            missing
                        )
                    )
            }
        )

failure_df = pd.DataFrame(
    failures
)

print(
    "Incomplete positive S1 entities:",
    f"{len(failure_df):,}"
)

if len(failure_df):

    print(
        "\nFirst 20 failures:"
    )

    print(
        failure_df.head(20).to_string(
            index=False
        )
    )

else:

    print(
        "No incomplete positive S1 entities."
    )


# ============================================================
# 18. SAVE RESULTS
# ============================================================

print("\n" + "=" * 100)
print("STEP 14 — SAVE RESULTS")
print("=" * 100)

threshold_results_df.to_csv(
    OUTPUT_DIR
    / "m2_2_threshold_results.csv",
    index=False
)

incremental_results_df.to_csv(
    OUTPUT_DIR
    / "m2_2_incremental_results.csv",
    index=False
)

block_results_df.to_csv(
    OUTPUT_DIR
    / "m2_2_block_results.csv",
    index=False
)

oracle_df.to_csv(
    OUTPUT_DIR
    / "m2_2_oracle_results.csv",
    index=False
)

failure_df.to_csv(
    OUTPUT_DIR
    / "m2_2_failures.csv",
    index=False
)


# ============================================================
# 19. SAVE BEST CANDIDATE SET
# ============================================================

best_candidates_df = pd.DataFrame(
    list(best_union_pairs),
    columns=[
        "s1_id",
        "target_id"
    ]
)

best_candidates_df = (
    best_candidates_df
    .sort_values(
        [
            "s1_id",
            "target_id"
        ]
    )
    .reset_index(drop=True)
)

best_candidate_path = (
    OUTPUT_DIR
    / "candidate_pairs_m2_2_best.tsv"
)

best_candidates_df.to_csv(
    best_candidate_path,
    sep="\t",
    index=False
)

print(
    "Best candidates saved:",
    best_candidate_path
)


# ============================================================
# 20. GENERATE REPORT
# ============================================================

report = []

report.append(
    "# M2.2 — Rare Token + Country-Aware Blocking\n\n"
)

report.append(
    "## Configuration\n\n"
)

report.append(
    f"- Random seed: **{SEED}**\n"
)

report.append(
    f"- S1 sample: **{len(s1_sample):,}**\n"
)

report.append(
    f"- Ground-truth pairs: **{len(gt_pairs):,}**\n"
)

report.append(
    f"- DF thresholds: **{DF_THRESHOLDS}**\n"
)

report.append(
    "- Country-aware token partition: **Yes**\n"
)

report.append(
    "- Candidate cap: **None**\n\n"
)

report.append(
    "## M2.1 Baseline\n\n"
)

report.append(
    f"- Candidate pairs: **{len(m21_pair_set):,}**\n"
)

report.append(
    f"- Pair recall: **{m21_recall:.6%}**\n\n"
)

report.append(
    "## Threshold Results\n\n"
)

report.append(
    threshold_results_df.to_markdown(
        index=False
    )
)

report.append(
    "\n\n## Incremental Results\n\n"
)

report.append(
    incremental_results_df.to_markdown(
        index=False
    )
)

report.append(
    "\n\n## Block Results\n\n"
)

report.append(
    block_results_df.to_markdown(
        index=False
    )
)

report.append(
    "\n\n## Oracle Results\n\n"
)

report.append(
    oracle_df.to_markdown(
        index=False
    )
)

report.append(
    "\n\n## Best Configuration\n\n"
)

report.append(
    f"- DF threshold: **{BEST_THRESHOLD}**\n"
)

report.append(
    f"- Candidate pairs: **{len(best_union_pairs):,}**\n"
)

report.append(
    f"- Pair recall: **{best_row['pair_recall']:.6%}**\n"
)

report.append(
    f"- Oracle macro F0.5: "
    f"**{best_row['oracle_macro_f05']:.6%}**\n"
)

report.append(
    f"- Incomplete positive S1: "
    f"**{len(failure_df):,}**\n"
)

report_path = (
    REPORT_DIR
    / "M2_2_RARE_TOKEN_RESULTS.md"
)

report_path.write_text(
    "".join(report),
    encoding="utf-8"
)

print(
    "Report saved:",
    report_path
)


# ============================================================
# 21. FINAL DASHBOARD
# ============================================================

print("\n\n")
print("=" * 110)
print("M2.2 FINAL DASHBOARD")
print("=" * 110)

print("\nM2.1 BASELINE")

print(
    f"Pair recall       : {m21_recall:.6%}"
)

print(
    f"Candidate pairs   : {len(m21_pair_set):,}"
)

print("\nM2.2 THRESHOLD COMPARISON")

dashboard_cols = [
    "df_threshold",
    "rare_pair_recall",
    "new_true_pairs_recovered",
    "union_pair_recall",
    "entity_complete_recall",
    "union_candidate_pairs",
    "mean_candidates",
    "p95_candidates",
    "p99_candidates"
]

print(
    threshold_results_df[
        dashboard_cols
    ].to_string(
        index=False
    )
)

print("\nORACLE F0.5")

print(
    oracle_df.to_string(
        index=False
    )
)

print("\nBEST CONFIGURATION")

print(
    f"DF threshold           : {BEST_THRESHOLD}"
)

print(
    f"Union candidates       : "
    f"{len(best_union_pairs):,}"
)

print(
    f"Pair recall            : "
    f"{best_row['pair_recall']:.6%}"
)

print(
    f"Oracle macro F0.5      : "
    f"{best_row['oracle_macro_f05']:.6%}"
)

print(
    f"Incomplete positive S1 : "
    f"{len(failure_df):,}"
)

print(
    "\nTotal runtime:",
    f"{time.time() - overall_start:.2f}s"
)

print("\nOUTPUT FILES")

print(
    OUTPUT_DIR
    / "m2_2_threshold_results.csv"
)

print(
    OUTPUT_DIR
    / "m2_2_incremental_results.csv"
)

print(
    OUTPUT_DIR
    / "m2_2_block_results.csv"
)

print(
    OUTPUT_DIR
    / "m2_2_oracle_results.csv"
)

print(
    OUTPUT_DIR
    / "m2_2_failures.csv"
)

print(
    best_candidate_path
)

print(
    report_path
)

print("\n" + "=" * 110)
print("M2.2 COMPLETE")
print("=" * 110)
print(
    "Do NOT start M2.3 yet."
)
print(
    "Review the dashboard and oracle ceiling first."
)
print("=" * 110)